# Extraction des zones vers Heurist

## Concatène images avec info

In [28]:
import re, os, csv, io, json
import pandas as pd
import glob



# --- Config à ajouter ---
ZONES_INPUT_DIR = "../List-of-zones"
ZONES_GLOB_PATTERN = "*_labelstudio.csv"
IMAGES_CSV_PATH = "../List-of-images/JJ096-JJ211_image_data_with_resolved_urls.csv"  

ARK_ID_RE = re.compile(r"ark:/\d+/[A-Za-z0-9]+")


In [47]:
# --- Table de correspondance des noms de colonnes alternatifs ---
COLUMN_ALIASES = {
    "registre": "register",
    "ordre": "folio_sort_key",
    # ajoutez ici d'autres variantes repérées (ex: "folio": "folio_label")
}


def normalize_zone_columns(df: pd.DataFrame, source_file: str) -> pd.DataFrame:
    """Renomme les colonnes selon COLUMN_ALIASES (comparaison insensible à la casse/espaces)."""
    rename_map = {}
    for col in df.columns:
        key = col.strip().lower()
        if key in COLUMN_ALIASES:
            rename_map[col] = COLUMN_ALIASES[key]
    if rename_map:
        print(f"  Normalisation colonnes ({source_file}): {rename_map}")
        df = df.rename(columns=rename_map)
    return df

def check_missing_canonical_columns(zones_df: pd.DataFrame,
                                     expected=("register", "folio_label", "folio_sort_key",
                                               "image", "image_path", "label")):
    """Signale, par fichier source, les colonnes canoniques absentes après normalisation."""
    for source_file, group in zones_df.groupby("__source_file"):
        missing = [c for c in expected if c not in zones_df.columns or group[c].isna().all()]
        if missing:
            print(f"⚠️  {source_file}: colonnes manquantes/vides après normalisation: {missing}")
            

def sniff_delimiter(sample_text: str) -> str:
    """Détecte le séparateur (tabulation, virgule, point-virgule...) à partir d'un échantillon."""
    try:
        dialect = csv.Sniffer().sniff(sample_text, delimiters="\t,;|")
        return dialect.delimiter
    except csv.Error:
        # Par défaut on suppose une tabulation (format observé dans les exemples fournis)
        return "\t"


def extract_ark_id(url: str):
    """Extrait l'identifiant ark:/xxxxx/yyyyy d'une URL IIIF, indépendamment du suffixe de taille."""
    if not isinstance(url, str):
        return None
    m = ARK_ID_RE.search(url)
    return m.group(0) if m else None


def extract_base_image_id(physical_url: str):
    """
    À partir d'une URL image physique IIIF (.../DEPOT/{base_image}/{region}/{size}/{rotation}/{quality}.{ext}),
    extrait l'identifiant {base_image} (ex. FRCHANJJ_JJ037_0004R_A).
    """
    if not isinstance(physical_url, str):
        return None
    parts = physical_url.rstrip("/").split("/")
    if len(parts) < 5:
        return None
    return parts[-5]


def read_zone_export(filepath: str) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8", errors="strict") as f:
        text = f.read()
    delimiter = sniff_delimiter(text[:5000])
    df = pd.read_csv(io.StringIO(text), sep=delimiter, dtype=str, keep_default_na=False, na_values=[""])
    df = normalize_zone_columns(df, os.path.basename(filepath))
    df["__source_file"] = os.path.basename(filepath)
    return df


def load_all_zone_exports(input_dir: str, pattern: str) -> pd.DataFrame:
    """Concatène tous les exports LS d'un dossier."""
    filepaths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if not filepaths:
        raise FileNotFoundError(f"Aucun export LS trouvé dans '{input_dir}' avec le motif '{pattern}'.")
    frames = []
    for fp in filepaths:
        print(f"Lecture export LS: {fp}")
        frames.append(read_zone_export(fp))
    combined = pd.concat(frames, ignore_index=True, sort=False)
    print(f"\nTotal: {len(frames)} fichiers, {len(combined)} lignes (images annotées).")
    return combined

def read_images_csv(filepath: str) -> pd.DataFrame:
    """
    Lit le CSV des images déjà fusionné et résolu (sortie du premier notebook,
    colonnes urlImage_arkId / urlImage). Écrit en UTF-8, séparateur ',' par défaut
    (pandas.to_csv standard).
    """
    df = pd.read_csv(filepath, dtype=str, keep_default_na=False, na_values=[""])

    required_cols = {"urlImage_arkId", "urlImage"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Colonnes manquantes dans {filepath}: {missing}. "
            f"Est-ce bien le fichier images_merged_with_resolved_urls.csv (sortie du 1er notebook) ?"
        )

    print(f"{len(df)} lignes lues depuis {filepath}.")
    n_missing_physical = (df["urlImage"] == "").sum() if "" in df["urlImage"].values else df["urlImage"].isna().sum()
    if n_missing_physical:
        print(f"⚠️  {n_missing_physical} ligne(s) sans URL physique résolue (urlImage vide).")
    return df


def build_unique_images_from_zones(zones_df: pd.DataFrame, images_df: pd.DataFrame,
                                    include_unannotated: bool = True):
    df = zones_df.copy()
    df["ark_id"] = df["image"].apply(extract_ark_id)

    dup_mask = df.duplicated(subset=["register", "ark_id"], keep=False)
    if dup_mask.any():
        n_dup = dup_mask.sum()
        print(f"⚠️  ATTENTION: {n_dup} lignes en doublon détectées sur (register, ark_id) !")
        print(df.loc[dup_mask, ["register", "ark_id", "image_path", "__source_file"]].to_string(index=False))
    else:
        print("Aucun doublon (register, ark_id) détecté.")

    df["n_zones"] = df["label"].apply(lambda x: len(json.loads(x)) if isinstance(x, str) and x.strip() else 0)
    if not include_unannotated:
        before = len(df)
        df = df[df["n_zones"] > 0].copy()
        print(f"Filtrage images sans zone: {before} -> {len(df)} lignes.")

    # --- Jointure élargie : on récupère aussi imageLabel comme repli pour folio_label ---
    images_lookup = images_df.copy()
    images_lookup["ark_id"] = images_lookup["urlImage_arkId"].apply(extract_ark_id)
    images_lookup = images_lookup.drop_duplicates(subset=["ark_id"])[["ark_id", "urlImage", "imageLabel"]]

    df = df.merge(images_lookup, on="ark_id", how="left", suffixes=("", "_from_images_list"))

    # --- Complète folio_label manquant/vide avec imageLabel (liste d'images) ---
    if "folio_label" not in df.columns:
        df["folio_label"] = pd.NA
    missing_folio = df["folio_label"].isna() | (df["folio_label"].astype(str).str.strip() == "")
    n_filled = (missing_folio & df["imageLabel"].notna() & (df["imageLabel"].astype(str).str.strip() != "")).sum()
    df.loc[missing_folio, "folio_label"] = df.loc[missing_folio, "imageLabel"]
    if n_filled:
        print(f"folio_label complété depuis imageLabel pour {n_filled} ligne(s).")
    still_missing = (df["folio_label"].isna() | (df["folio_label"].astype(str).str.strip() == "")).sum()
    still_missing_mask = df["folio_label"].isna() | (df["folio_label"].astype(str).str.strip() == "")
    still_missing = still_missing_mask.sum()
    if still_missing:
        print(f"⚠️  {still_missing} ligne(s) encore sans folio_label (ni zones ni images).")
        cols = [c for c in ["register", "ark_id", "image_path", "folio_sort_key", "__source_file"] if c in df.columns]
        print(df.loc[still_missing_mask, cols].to_string(index=False))
        
    # --- Suppression des images sans correspondance physique, puis reset de l'index ---
    n_before = len(df)
    unmatched_df = df[df["urlImage"].isna() | (df["urlImage"] == "")].copy()
    df = df[df["urlImage"].notna() & (df["urlImage"] != "")].reset_index(drop=True)
    n_removed = n_before - len(df)
    if n_removed:
        print(f"🗑️  {n_removed} image(s) sans correspondance supprimée(s) de la liste.")

    df["base_image"] = df["urlImage"].apply(extract_base_image_id)

    
    # --- ID temporaire, préfixé par registre ---
    df["temp_id1"] = (
        df.groupby("register").cumcount().add(1).astype(str).str.zfill(4)
    )
    df["temp_id1"] = df["register"] + "_" + df["temp_id1"]

     # --- ID temporaire : entier séquentiel simple, 1 à N ---
    df = df.reset_index(drop=True)
    df["temp_id2"] = df.index + 1

 
    final_cols = [
        "temp_id2","temp_id1", "register", "folio_label", "folio_sort_key",
        "ark_id", "urlImage", "base_image", "image_path", "n_zones", "__source_file",
    ]
    return df[final_cols], unmatched_df


def print_unmatched_images(df: pd.DataFrame) -> pd.DataFrame:
    """Affiche origine et identité des images qui n'ont pas trouvé d'URL physique."""
    unmatched = df[df["urlImage"].isna() | (df["urlImage"] == "")]
    cols = [c for c in ["register", "folio_label", "folio_sort_key", "ark_id",
                         "image_path", "__source_file"] if c in unmatched.columns]
    print(f"{len(unmatched)} image(s) sans correspondance :")
    print(unmatched[cols].to_string(index=False))
    return unmatched

In [48]:
zones_df = load_all_zone_exports(ZONES_INPUT_DIR, ZONES_GLOB_PATTERN)
check_missing_canonical_columns(zones_df)

unique_images_df, unmatched_df = build_unique_images_from_zones(zones_df, images_df, include_unannotated=True)
print(f"\n{len(unique_images_df)} images conservées, {len(unmatched_df)} écartées (voir unmatched_df pour le détail).")

unique_images_df.to_csv("images_a_importer_heurist.csv", index=False, encoding="utf-8")
unique_images_df.head()

Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ100-JJ118_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ100-JJ118_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ119-JJ132_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ119-JJ132_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ133-JJ139_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ133-JJ139_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ140-JJ159_labelstudio.

,temp_id2,temp_id1,register,folio_label,folio_sort_key,ark_id,urlImage,base_image,image_path,n_zones,__source_file
0,1,JJ096_0001,JJ096,plat sup��rieur,1,ark:/63955/vd0qz1ihxyni,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0746_A,images_registres_AN_JJ035_JJ211/images\Paris_A...,0,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
1,2,JJ096_0002,JJ096,contre-plat sup��rieur,2,ark:/63955/v6e576jlpbid,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0746_AB_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
2,3,JJ096_0003,JJ096,1r,3,ark:/63955/vxyc3rg33kh4,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0747_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
3,4,JJ096_0004,JJ096,1v,4,ark:/63955/vbg5uy3smlz0,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0748_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
4,5,JJ096_0005,JJ096,2r,5,ark:/63955/vr66brwmaaj8,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0749_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...


zones_df = load_all_zone_exports(ZONES_INPUT_DIR, ZONES_GLOB_PATTERN)
check_missing_canonical_columns(zones_df)


images_df = read_images_csv(IMAGES_CSV_PATH)

unique_images_df = build_unique_images_from_zones(zones_df, images_df, include_unannotated=True)
unmatched_df = print_unmatched_images(unique_images_df)
unique_images_df.to_csv("images_a_importer_heurist.csv", index=False, encoding="utf-8")
unique_images_df.head()

## Explose les zones

In [53]:
import json

def explode_zone_rectangles(zones_df: pd.DataFrame, images_df: pd.DataFrame,
                             unique_images_df: pd.DataFrame,
                             thumbnail_width: int = 600) -> pd.DataFrame:
    """
    Explose la colonne JSON 'label' de chaque ligne zones_df (une image) en une ligne
    par rectangle annoté, avec coordonnées pixel (image pleine résolution), URL de
    vignette IIIF recadrée, et référence à l'image (temp_id) pour liaison ultérieure.
    """
    # Dimensions réelles + URL physique, indexées par ark_id
    dims_lookup = images_df.copy()
    dims_lookup["ark_id"] = dims_lookup["urlImage_arkId"].apply(extract_ark_id)
    dims_lookup["imageWidthAsDownloaded"] = pd.to_numeric(dims_lookup["imageWidthAsDownloaded"], errors="coerce")
    dims_lookup["imageHeightAsDownloaded"] = pd.to_numeric(dims_lookup["imageHeightAsDownloaded"], errors="coerce")
    dims_lookup = dims_lookup.drop_duplicates(subset=["ark_id"])[
        ["ark_id", "urlImage", "imageWidthAsDownloaded", "imageHeightAsDownloaded"]
    ]

    # temp_id des images (référence pour les zones)
    image_ids_lookup = unique_images_df[["ark_id", "temp_id2"]].rename(columns={"temp_id2": "image_temp_id"})

    df = zones_df.copy()
    df["ark_id"] = df["image"].apply(extract_ark_id)
    df = df.merge(dims_lookup, on="ark_id", how="left")
    df = df.merge(image_ids_lookup, on="ark_id", how="left")

    zone_rows = []
    n_parse_errors = 0
    n_missing_dims = 0

    for row in df.itertuples(index=False):
        label_raw = getattr(row, "label")
        if not isinstance(label_raw, str) or not label_raw.strip():
            continue  # image sans zone

        try:
            rectangles = json.loads(label_raw)
        except (json.JSONDecodeError, TypeError):
            n_parse_errors += 1
            continue

        full_w = getattr(row, "imageWidthAsDownloaded")
        full_h = getattr(row, "imageHeightAsDownloaded")
        physical_url = getattr(row, "urlImage")

        if pd.isna(full_w) or pd.isna(full_h) or not isinstance(physical_url, str) or not physical_url:
            n_missing_dims += 1
            continue

        # URL racine IIIF (sans le suffixe /region/size/rotation/quality.ext)
        iiif_root = "/".join(physical_url.rstrip("/").split("/")[:-4])

        for order, rect in enumerate(rectangles, start=1):
            x_pct = rect.get("x", 0)
            y_pct = rect.get("y", 0)
            w_pct = rect.get("width", 0)
            h_pct = rect.get("height", 0)
            part_labels = rect.get("rectanglelabels", [])
            part = part_labels[0] if part_labels else None

            x_px = round(x_pct / 100 * full_w)
            y_px = round(y_pct / 100 * full_h)
            w_px = round(w_pct / 100 * full_w)
            h_px = round(h_pct / 100 * full_h)
            x2_px, y2_px = x_px + w_px, y_px + h_px

            coordinates = f"{x_px},{y_px} {x2_px},{y_px} {x2_px},{y2_px} {x_px},{y2_px}"
            crop_path = f"{x_px},{y_px},{w_px},{h_px}/{thumbnail_width},/0/default.jpg"
            address_bvmm_path = f"{iiif_root}/{crop_path}"

            zone_rows.append({
                "register": getattr(row, "register", None),
                "folio_label": getattr(row, "folio_label", None),
                "folio_sort_key": getattr(row, "folio_sort_key", None),
                "zone_order_on_folio": order,
                "part": part,
                "coordinates": coordinates,
                "base_image": extract_base_image_id(physical_url),
                "address_bvmm_path": address_bvmm_path,
                "address_bvmm_name": "_remote",
                "image_ark_id": getattr(row, "ark_id"),
                "image_temp_id": getattr(row, "image_temp_id"),
                "__source_file": getattr(row, "__source_file", None),
            })

    zones_out = pd.DataFrame(zone_rows)
    zones_out.insert(0, "zone_temp_id", zones_out.index + 1)

    print(f"{len(zones_out)} zones extraites depuis {len(df)} images.")
    if n_parse_errors:
        print(f"⚠️  {n_parse_errors} ligne(s) avec un JSON 'label' invalide (ignorées).")
    if n_missing_dims:
        print(f"⚠️  {n_missing_dims} ligne(s) sans dimensions/URL physique (zones ignorées).")

    return zones_out

In [56]:
zones_exploded_df = explode_zone_rectangles(zones_df, images_df, unique_images_df)
zones_exploded_df.head()
zones_exploded_df.to_csv("all_zones.csv")

83964 zones extraites depuis 49849 images.
⚠️  2 ligne(s) sans dimensions/URL physique (zones ignorées).


In [ ]:
!label-studio